# Preparación de datos para el predictor de género

Este notebook convierte los hallazgos del EDA en un proceso reproducible. No entrena un modelo y no elige hiperparámetros.

El EDA se calculó sobre las 679.807 filas, antes de separar train y test. Esas decisiones quedan fijadas aquí. A partir del split, el conjunto de test no se usa para elegir ni para ajustar transformaciones.

## 1. Propósito y contexto

**Unidad de observación.** Una pista: una fila, un `id`, un string `chords`.

**Variable objetivo.** `main_genre`, una de 12 clases. Es un atributo del artista en Spotify, copiado a cada pista de ese artista.

**Qué es una predicción.** El género de una canción cuyo artista no aparece en el entrenamiento. La entrada es el string `chords`: símbolos de acorde y, cuando la transcripción los trae, tags de parte (`<verse_1>`, `<chorus_2>`, …).

**Qué estaría disponible al predecir.** Solo esa progresión. `artist_id` sirve para agrupar el split y no entra como feature. `genres` es la lista de tags de la que se mapeó `main_genre`. `rock_genre` es otra tarea (179 clases). `release_date` y `decade` no forman parte de este escenario: discrepan en el 12% de las pistas fechadas, y la década es un candidato débil (la masa etiquetada es pop/rock reciente).

**Columnas que no llegan al modelo**

| Columna | Por qué se excluye |
| --- | --- |
| `id`, `spotify_song_id`, `spotify_artist_id` | Identificadores y enlaces |
| `artist_id` | Clave del split. Como feature memoriza al artista y, con él, la etiqueta |
| `genres` | Fuente de la que se construyó `main_genre` |
| `rock_genre` | Otra taxonomía, ausente en la mayoría de las filas |
| `release_date`, `decade` | No son la entrada del escenario, y no son intercambiables |
| `main_genre` | Es el target |
| `chords` (texto crudo) | Se consume al construir features y no se conserva como string |


## 2. Hallazgos del EDA que esta preparación usa

No se repite el análisis. Estos son los resultados que justifican las transformaciones:

- De 679.807 pistas, 352.111 tienen `main_genre`. Las 77.642 con tags de Spotify y sin `main_genre` (2.017 tags; las 30 más frecuentes cubren el 37%) quedan fuera del problema de 12 clases.
- Etiquetadas y no etiquetadas tienen progresiones parecidas (mediana de 72 frente a 69 acordes; mediana de 6 acordes únicos). Filtrar por etiqueta no selecciona otro tipo de canción. Toda fila etiquetada tiene `artist_id`.
- El 96,4% de las etiquetadas pertenece a un artista con dos o más pistas (mediana 3). Un split aleatorio por fila reparte la etiqueta.
- Hay 2.509 grupos de progresiones idénticas (5.117 filas). 110 grupos (267 filas) cruzan artistas. El split por artista no absorbe esas copias.
- `chords` está completo. El resto de los metadatos falta en bloque (el join con Spotify), no al azar.
- La longitud tiene cola derecha: mediana cerca de 71, p99 = 201, máximo 2.549. En la cola predominan loops repetidos.
- El conteo de acordes únicos separa más que la longitud (jazz y metal más amplios; pop y rock siguen cerca).
- La mezcla de calidad cambia por género como proporción. El parser deja `unknown` en 3 tokens. No aparecen grafías de medio disminuido; `dim7` cuenta como disminuido.
- Los tags de parte marcan forma. En 175.956 pistas con verse y chorus, la mezcla de calidad difiere en alrededor de un punto porcentual. No se construye un contraste armónico entre secciones.
- Los loops ABAB son frecuentes (8,5% de las ventanas de pop, 12,8% de rock, 18,8% de reggae).
- Dentro de pop / rock / alternative / punk, el coseno baja de 0,988 (bolsa de nombres) a 0,945 (4-gramas) y queda en 0,943 con grados de la escala. El margen de tonalidad es menor que 0,05 en el 29,6% de las pistas. Esas representaciones quedan para comparar en el notebook de modelado.
- Pop es cerca del 24% de las etiquetadas. La métrica del modelado deberá incluir macro-F1. Aquí no se rebalancea.


## 3. Del EDA a las decisiones

| Hallazgo del EDA | Evidencia | Decisión | Justificación |
| --- | --- | --- | --- |
| La etiqueta existe solo en la mitad de las filas | 352.111 con `main_genre`; 77.642 con tags no mapeados | Entrenar solo con filas etiquetadas | Esas filas no son una clase «other» |
| La etiqueta es del artista | 96,4% de las etiquetadas están en un `artist_id` con 2+ pistas; todas tienen `artist_id` | `GroupShuffleSplit` por `artist_id`, 20% de artistas en test, semilla 42 | Un split por fila pone al mismo artista, y la misma etiqueta, en los dos lados. No se estratifica por fila porque un artista no se parte |
| Progresiones repetidas entre artistas | 110 grupos, 267 filas | No descartarlas en esta etapa | El split por artista ya cubre las copias del mismo artista. Tirar las otras 267 filas sería una decisión nueva; no se mira el test para tomarla |
| Cola derecha de la longitud | p99 = 201, máximo 2.549, sobre todo loops | Conservar `n_chords` y `log1p_n_chords` | `log1p` reduce la asimetría sin borrar filas. El conteo crudo se conserva porque el modelado aún no elige entre un modelo lineal y uno de árbol |
| El vocabulario único varía más que la longitud | Jazz y metal más amplios; pop y rock cercanos | `n_unique_chords` y `unique_ratio` | Es una candidata del EDA, con el límite de que no separa pop de rock |
| La calidad cambia como proporción | Más power chords en metal, más menor en jazz; `unknown` = 3 tokens | Proporciones de calidad por pista | El parser ya está comprobado. No se añade una columna de medio disminuido: en el corpus esa grafía no aparece |
| Los tags marcan forma, no armonía | Verse y chorus difieren ~1 punto en la mezcla de calidad | `has_structure` como bandera binaria | Estaría disponible en el tab. No se resta la calidad del estribillo a la de la estrofa |
| Los loops ABAB dependen del género | 8,5% pop, 12,8% rock, 18,8% reggae | `abab_share` | El paper los descarta al contar 4-gramas; este EDA pide conservarlos |
| Tiempo débil e inconsistente | Década sesgada a 2010–2020; 12% de desacuerdo con `release_date` | Excluir `decade` y `release_date` | El escenario de predicción lee `chords`. Usar los dos campos a la vez repetiría un dato que no coincide |
| Bolsa, n-gramas y grados siguen abiertos | Coseno del bloque pop/rock: 0,988 → 0,945 → 0,943; margen de tonalidad < 0,05 en el 29,6% | No materializar esas representaciones aquí | Ajustar un vocabulario ahora elegiría una de ellas y dispararía la dimensión. La comparación queda para el notebook de modelado |
| Desbalance | Pop ≈ 24% | No remuestrear | Cambia la pregunta de modelado. La métrica (macro-F1) se decide después |


## 4. Imports, constantes y rutas

El notebook no usa variables del EDA. La ruta de los CSV se resuelve igual que allí: `data/` si el kernel está en la raíz del repositorio, `../data/` si está en `notebooks/`.


In [1]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
TEST_SIZE = 0.20
YEAR_MIN = 1899
YEAR_MAX = 2023

REQUIRED_COLUMNS = [
    "id",
    "chords",
    "release_date",
    "genres",
    "decade",
    "rock_genre",
    "artist_id",
    "main_genre",
    "spotify_song_id",
    "spotify_artist_id",
]

MAIN_GENRES = frozenset(
    {
        "pop",
        "rock",
        "country",
        "alternative",
        "pop rock",
        "punk",
        "metal",
        "rap",
        "soul",
        "jazz",
        "reggae",
        "electronic",
    }
)

# Identidad, fuente de la etiqueta, otra tarea, tiempo y el target.
EXCLUDED_COLUMNS = [
    "id",
    "artist_id",
    "spotify_song_id",
    "spotify_artist_id",
    "genres",
    "rock_genre",
    "release_date",
    "decade",
    "main_genre",
    "chords",
]

if (Path("data") / "chordonomicon_v2_part_0.csv").exists():
    DATA_DIR = Path("data")
elif (Path("..") / "data" / "chordonomicon_v2_part_0.csv").exists():
    DATA_DIR = Path("..") / "data"
else:
    raise FileNotFoundError(
        "No están los CSV chordonomicon_v2_part_*.csv en data/ ni en ../data/."
    )

print("DATA_DIR =", DATA_DIR.resolve())
print("semilla =", SEED, "| fracción de artistas en test =", TEST_SIZE)


DATA_DIR = /home/grivas/Escritorio/music_Chordonomicon_genre_predictor/data
semilla = 42 | fracción de artistas en test = 0.2


## 5. Carga y validación

Los siete CSV son un corte de almacenamiento, no un split de modelado. Se concatenan y después se validan. Si el esquema, los identificadores, las clases o un año fechado no cuadran, la celda se detiene con un mensaje explícito.


In [2]:
def load_shards(data_dir: Path) -> pd.DataFrame:
    paths = sorted(data_dir.glob("chordonomicon_v2_part_*.csv"))
    if not paths:
        raise FileNotFoundError(
            f"No hay chordonomicon_v2_part_*.csv en {data_dir.resolve()}."
        )
    frames: list[pd.DataFrame] = []
    header: list[str] | None = None
    for path in paths:
        # Un solo pase de tipos. Con el lector por trozos, un bloque de nulos
        # se infiere como número y el siguiente como texto.
        part = pd.read_csv(path, low_memory=False)
        columns = list(part.columns)
        if header is None:
            header = columns
        elif columns != header:
            raise ValueError(
                f"{path.name} no tiene el mismo encabezado que {paths[0].name}: {columns}"
            )
        frames.append(part)
        print(f"{path.name}: {len(part):,} filas")
    return pd.concat(frames, ignore_index=True)


def validate_raw(df: pd.DataFrame) -> None:
    missing = [column for column in REQUIRED_COLUMNS if column not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    duplicated_ids = int(df["id"].duplicated().sum())
    if duplicated_ids:
        raise ValueError(f"Hay {duplicated_ids:,} id duplicados. Cada fila debe ser una pista.")

    if df["chords"].isna().any():
        raise ValueError(
            "chords tiene valores ausentes. En el EDA esa columna está completa y es la entrada del modelo."
        )

    present = set(df["main_genre"].dropna().unique())
    unexpected = sorted(present - MAIN_GENRES)
    if unexpected:
        raise ValueError(f"main_genre tiene clases fuera de las 12 esperadas: {unexpected}")
    absent = sorted(MAIN_GENRES - present)
    if absent:
        raise ValueError(f"main_genre no contiene estas clases esperadas: {absent}")

    dated = df["release_date"].notna()
    years = pd.to_numeric(
        df.loc[dated, "release_date"].astype("string").str.slice(0, 4),
        errors="coerce",
    )
    unreadable = int(years.isna().sum())
    if unreadable:
        raise ValueError(
            f"{unreadable:,} release_date no empiezan por un año de cuatro dígitos."
        )
    out_of_range = (years < YEAR_MIN) | (years > YEAR_MAX)
    n_out = int(out_of_range.sum())
    if n_out:
        sample = years.loc[out_of_range].head(5).tolist()
        raise ValueError(
            f"{n_out:,} fechas tienen año fuera de {YEAR_MIN}–{YEAR_MAX}. Ejemplos: {sample}"
        )

    numeric = df.select_dtypes(include="number")
    n_inf = int(np.isinf(numeric.to_numpy(dtype=float)).sum())
    if n_inf:
        raise ValueError(f"Hay {n_inf:,} valores infinitos en columnas numéricas.")

    print("Validación superada.")
    print(f"Filas: {len(df):,} | columnas: {list(df.columns)}")
    print(f"id únicos: {df['id'].nunique():,}")
    print(f"Con main_genre: {int(df['main_genre'].notna().sum()):,}")
    print(f"Años fechados dentro de {YEAR_MIN}–{YEAR_MAX}: {int(dated.sum()):,}")


raw = load_shards(DATA_DIR)
validate_raw(raw)


chordonomicon_v2_part_0.csv: 100,000 filas


chordonomicon_v2_part_1.csv: 100,000 filas


chordonomicon_v2_part_2.csv: 100,000 filas


chordonomicon_v2_part_3.csv: 100,000 filas


chordonomicon_v2_part_4.csv: 100,000 filas


chordonomicon_v2_part_5.csv: 100,000 filas


chordonomicon_v2_part_6.csv: 79,807 filas


Validación superada.
Filas: 679,807 | columnas: ['id', 'chords', 'release_date', 'genres', 'decade', 'rock_genre', 'artist_id', 'main_genre', 'spotify_song_id', 'spotify_artist_id']
id únicos: 679,807
Con main_genre: 352,111
Años fechados dentro de 1899–2023: 422,181


## 6. Separación train / test

Primero se deja el universo supervisado: filas con `main_genre`. El EDA midió que todas esas filas tienen `artist_id`; si eso dejara de cumplirse, el split agrupado no estaría definido y la celda se detiene.

`GroupShuffleSplit` aparta un 20% de los artistas, no un 20% exacto de las filas. La semilla fija el corte. No se estratifica por género: partir un artista repartiría su etiqueta entre train y test.

El test se describe (tamaños y conteos por clase) y no se vuelve a consultar para construir features, imputar ni escalar.


In [3]:
supervised = raw.loc[raw["main_genre"].notna()].copy()
missing_artist = int(supervised["artist_id"].isna().sum())
if missing_artist:
    raise ValueError(
        f"{missing_artist:,} filas tienen main_genre y no tienen artist_id. "
        "El split agrupado no puede asignarlas sin romper el grupo."
    )

print(f"Universo supervisado: {len(supervised):,} filas, {supervised['artist_id'].nunique():,} artistas")

features = supervised.drop(columns=["main_genre"])
target = supervised["main_genre"]
groups = supervised["artist_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(features, target, groups))

X_train = features.iloc[train_idx].copy()
X_test = features.iloc[test_idx].copy()
y_train = target.iloc[train_idx].copy()
y_test = target.iloc[test_idx].copy()

shared_artists = set(X_train["artist_id"]).intersection(X_test["artist_id"])
shared_ids = set(X_train["id"]).intersection(X_test["id"])
if shared_artists:
    raise ValueError(f"El split comparte {len(shared_artists):,} artist_id entre train y test.")
if shared_ids:
    raise ValueError(f"El split comparte {len(shared_ids):,} id entre train y test.")

print(f"Train: {len(X_train):,} filas | {X_train['artist_id'].nunique():,} artistas")
print(f"Test:  {len(X_test):,} filas | {X_test['artist_id'].nunique():,} artistas")
print("Artistas en común: 0 | id en común: 0")

class_counts = pd.DataFrame(
    {
        "train": y_train.value_counts(),
        "test": y_test.value_counts(),
    }
).fillna(0).astype(int)
class_counts["train_share"] = class_counts["train"] / class_counts["train"].sum()
class_counts["test_share"] = class_counts["test"] / class_counts["test"].sum()
class_counts = class_counts.sort_values("train", ascending=False)
class_counts


Universo supervisado: 352,111 filas, 38,057 artistas


Train: 283,147 filas | 30,445 artistas
Test:  68,964 filas | 7,612 artistas
Artistas en común: 0 | id en común: 0


,train,test,train_share,test_share
main_genre,,,,
pop,68696,16489,0.242616,0.239096
rock,55033,12205,0.194362,0.176976
country,42324,10982,0.149477,0.159243
alternative,37343,9909,0.131886,0.143684
pop rock,31675,7882,0.111868,0.114292
punk,12932,3134,0.045672,0.045444
metal,9160,2155,0.032351,0.031248
rap,8911,2275,0.031471,0.032988
soul,5887,1463,0.020791,0.021214


El corte queda descrito arriba. Las cuotas de test no modifican el pipeline: el desbalance entre géneros es el que el EDA ya midió, y un artista entero viaja a un solo lado.

Las 267 filas cuya progresión se repite en artistas distintos pueden quedar repartidas. Es leakage residual de la progresión, más chico que el de un split por fila. No se eliminan.


## 7. Feature engineering

Todas las features se calculan con `chords`, que está completo y existiría al predecir un tab. El transformer no aprende estadísticas del corpus: la misma función se aplica en train y en test. Por eso `fit` no estima medias, vocabularios ni tonalidades.

Familias:

1. Estadísticas por fila: `n_chords`, `log1p_n_chords`, `n_unique_chords`, `unique_ratio`.
2. Dominio armónico: proporción de cada calidad que el parser del EDA ya contrastó.
3. Forma y repetición: `has_structure` y `abab_share`.

`abab_share` usa la condición del EDA sobre ventanas de cuatro acordes: el primero igual al tercero y el segundo igual al cuarto. Eso cuenta el loop ABAB y también el caso AAAA. Si hay menos de cuatro acordes, la proporción es 0. Una progresión vacía produce ceros, no NaN.

No se construye el contraste verse−chorus, ni grados de la escala, ni n-gramas. Esas comparaciones siguen abiertas y no se cierran con el test.


In [4]:
PART_RE = re.compile(r"^<([A-Za-z]+)_(\d+)>$")
POWER_RE = re.compile(r"^[A-G]s?(?:no3|5)(?:d)?$", re.I)
_SUFFIX_CHUNKS = (
    "minmaj",
    "augmaj",
    "min7b5",
    "m7b5",
    "hmin",
    "hdim",
    "min",
    "maj",
    "dim",
    "aug",
    "sus",
    "add",
    "no3",
    "13",
    "11",
    "9",
    "7",
    "6",
    "5",
    "4",
    "2",
    "b9",
    "b7",
    "b",
    "s",
    "d",
)

# El medio disminuido se reconoce, como en el EDA, y no se emite:
# allí esas grafías no aparecen, así que la columna sería constante.
QUALITY_COLUMNS = (
    "share_major",
    "share_minor",
    "share_power",
    "share_suspended",
    "share_diminished",
    "share_augmented",
    "share_unknown",
)
_QUALITY_INDEX = {
    "major": 0,
    "minor": 1,
    "power": 2,
    "suspended": 3,
    "diminished": 4,
    "augmented": 5,
    "unknown": 6,
}

NUMERIC_FEATURES = [
    "n_chords",
    "log1p_n_chords",
    "n_unique_chords",
    "unique_ratio",
    *QUALITY_COLUMNS,
    "abab_share",
]
BINARY_FEATURES = ["has_structure"]
FEATURE_COLUMNS = NUMERIC_FEATURES + BINARY_FEATURES


def _recognized_suffix(rest: str) -> bool:
    # Misma lectura que el EDA: un resto no reconocido no se llama mayor.
    text = rest.lower()
    while text:
        for chunk in _SUFFIX_CHUNKS:
            if text.startswith(chunk):
                text = text[len(chunk) :]
                break
        else:
            return False
    return True


def chord_quality(token: str) -> str:
    # Raíz A–G, accidente s/b (sus no es sostenido). El slash usa el acorde a la izquierda.
    core = token.split("/", 1)[0]
    if not core or core[0] not in "ABCDEFG":
        return "unknown"
    rest = core[1:]
    if rest and rest[0] in "bB":
        rest = rest[1:]
    elif rest and rest[0] in "sS" and not rest[1:].lower().startswith("us"):
        rest = rest[1:]
    if not _recognized_suffix(rest):
        return "unknown"
    lowered = core.lower()
    if "sus" in lowered:
        return "suspended"
    if "aug" in lowered:
        return "augmented"
    if "hmin" in lowered or "hdim" in lowered or "min7b5" in lowered or "m7b5" in lowered:
        return "half-diminished"
    if "dim" in lowered:
        return "diminished"
    if "no3" in lowered or POWER_RE.match(core):
        return "power"
    if "min" in lowered:
        return "minor"
    return "major"


class ChordFeaturesTransformer(BaseEstimator, TransformerMixin):
    """Features por pista a partir de `chords`. No estima nada en `fit`."""

    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame) or "chords" not in X.columns:
            raise ValueError(
                "ChordFeaturesTransformer necesita un DataFrame con la columna 'chords'."
            )
        self.n_features_in_ = X.shape[1]
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame) or "chords" not in X.columns:
            raise ValueError(
                "ChordFeaturesTransformer necesita un DataFrame con la columna 'chords'."
            )
        chords = X["chords"].to_numpy()
        n_rows = len(chords)
        out = np.zeros((n_rows, len(FEATURE_COLUMNS)), dtype=np.float64)
        i_n = FEATURE_COLUMNS.index("n_chords")
        i_log = FEATURE_COLUMNS.index("log1p_n_chords")
        i_unique = FEATURE_COLUMNS.index("n_unique_chords")
        i_ratio = FEATURE_COLUMNS.index("unique_ratio")
        i_abab = FEATURE_COLUMNS.index("abab_share")
        i_structure = FEATURE_COLUMNS.index("has_structure")
        i_quality = FEATURE_COLUMNS.index("share_major")

        for row, raw in enumerate(chords):
            chord_tokens: list[str] = []
            has_structure = 0.0
            for token in str(raw).split():
                if PART_RE.match(token):
                    has_structure = 1.0
                else:
                    chord_tokens.append(token)
            n_tokens = len(chord_tokens)
            out[row, i_n] = n_tokens
            out[row, i_log] = np.log1p(n_tokens)
            out[row, i_unique] = len(set(chord_tokens))
            out[row, i_ratio] = (len(set(chord_tokens)) / n_tokens) if n_tokens else 0.0
            out[row, i_structure] = has_structure
            if n_tokens:
                counts = np.zeros(len(QUALITY_COLUMNS), dtype=np.float64)
                for token in chord_tokens:
                    quality = _QUALITY_INDEX.get(chord_quality(token))
                    if quality is not None:
                        counts[quality] += 1.0
                out[row, i_quality : i_quality + len(QUALITY_COLUMNS)] = counts / n_tokens
            if n_tokens >= 4:
                windows = n_tokens - 3
                abab = 0
                for start in range(windows):
                    if (
                        chord_tokens[start] == chord_tokens[start + 2]
                        and chord_tokens[start + 1] == chord_tokens[start + 3]
                    ):
                        abab += 1
                out[row, i_abab] = abab / windows
        return pd.DataFrame(out, index=X.index, columns=FEATURE_COLUMNS)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(FEATURE_COLUMNS, dtype=object)


demo = pd.DataFrame(
    {
        "chords": [
            "<verse_1> C F C F <chorus_1> G C G C",
            "C C C C C",
            "",
        ]
    }
)
ChordFeaturesTransformer().fit_transform(demo)


,n_chords,log1p_n_chords,n_unique_chords,unique_ratio,share_major,share_minor,share_power,share_suspended,share_diminished,share_augmented,share_unknown,abab_share,has_structure
0,8.0,2.197225,3.0,0.375,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,1.0
1,5.0,1.791759,1.0,0.200,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.0,0.000000,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


La primera fila del ejemplo tiene 8 acordes, 3 grafías distintas y tags de parte, así que `has_structure` es 1. Dos de sus cinco ventanas cumplen ABAB (`C F C F` y `G C G C`), y `abab_share` queda en 0,4. La segunda es un acorde mayor repetido: `unique_ratio` es 1/5 y todas las ventanas son AAAA, que la misma regla cuenta. La tercera está vacía y sale en ceros, sin NaN.

`n_chords` y `log1p_n_chords` son monótonas una de la otra. Se conservan las dos a propósito: la segunda es la versión para un modelo lineal, la primera sigue siendo legible para un árbol. El notebook de modelado puede quedarse con una.


## 8. Grupos de columnas

Después del transformer la matriz ya no tiene columnas crudas.

- **Numéricas:** conteos, `log1p_n_chords`, razón, proporciones de calidad y `abab_share`. Van a imputación (guarda) y a escalado.
- **Binaria:** `has_structure`. Se imputa con la moda y no se escala, para que siga siendo 0/1.
- **Nominales y ordinales:** no quedan. `genres`, `rock_genre` y `artist_id` se excluyeron. `decade` también, así que no hay una variable ordinal que codificar.
- **Excluidas:** las de la sección 1, más el string `chords` una vez leído.

En este diseño `chords` no tiene huecos y las features de una progresión vacía son 0. `SimpleImputer` queda dentro del pipeline porque es lo que aprendería una mediana o una moda si una fila futura llegara incompleta, y se ajusta solo con train. Sobre estos datos debe actuar como identidad.


In [5]:
print("Numéricas:", NUMERIC_FEATURES)
print("Binaria:", BINARY_FEATURES)
print("Excluidas, no salen del transformer:", EXCLUDED_COLUMNS)


Numéricas: ['n_chords', 'log1p_n_chords', 'n_unique_chords', 'unique_ratio', 'share_major', 'share_minor', 'share_power', 'share_suspended', 'share_diminished', 'share_augmented', 'share_unknown', 'abab_share']
Binaria: ['has_structure']
Excluidas, no salen del transformer: ['id', 'artist_id', 'spotify_song_id', 'spotify_artist_id', 'genres', 'rock_genre', 'release_date', 'decade', 'main_genre', 'chords']


## 9. Pipeline

El orden es el de los datos: string crudo, features de acordes, imputación y escalado, y un hueco `passthrough` donde el notebook de modelado colocará un estimador. No hay búsqueda de hiperparámetros.

`fit` se llama solo con train.


In [6]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
binary_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ]
)
preprocess = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("binary", binary_pipeline, BINARY_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)
pipeline = Pipeline(
    steps=[
        ("chord_features", ChordFeaturesTransformer()),
        ("preprocess", preprocess),
        ("classifier", "passthrough"),
    ]
)

train_before_fit = X_train.copy()
pipeline.fit(X_train, y_train)
pd.testing.assert_frame_equal(X_train, train_before_fit)
print("Ajustado solo con train. X_train no cambió.")
print(pipeline)


Ajustado solo con train. X_train no cambió.
Pipeline(steps=[('chord_features', ChordFeaturesTransformer()),
                ('preprocess',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['n_chords', 'log1p_n_chords',
                                                   'n_unique_chords',
                                                   'unique_ratio',
                                                   'share_major', 'share_minor',
                                                   'share_power',
                                                   'share_suspended',
                    

## 10. Verificaciones, solo con train

Se comprueba el número de filas, la ausencia de NaN e infinitos, que las columnas excluidas no estén entre los nombres de salida, y que la entrada siga igual después de transformar. Una muestra sin escalar enseña las features creadas. El test no entra en estas cuentas.


In [7]:
unscaled_sample = pipeline.named_steps["chord_features"].transform(X_train.head(3))
Z_train = pipeline.transform(X_train)
output_names = list(pipeline.named_steps["preprocess"].get_feature_names_out())

print("Filas de entrada:", len(X_train))
print("Filas de salida:", Z_train.shape[0])
print("Columnas crudas de entrada:", X_train.shape[1])
print("Features de salida:", Z_train.shape[1])
print("Nombres:", output_names)

if Z_train.shape[0] != len(X_train):
    raise ValueError(
        f"El pipeline devolvió {Z_train.shape[0]:,} filas y train tiene {len(X_train):,}."
    )
if Z_train.shape[1] != len(FEATURE_COLUMNS):
    raise ValueError(
        f"Se esperaban {len(FEATURE_COLUMNS)} features y salieron {Z_train.shape[1]}."
    )

matrix = np.asarray(Z_train, dtype=float)
n_bad = int((~np.isfinite(matrix)).sum())
if n_bad:
    raise ValueError(f"La matriz de train tiene {n_bad:,} NaN o infinitos.")

bare_names = [name.split("__", 1)[-1] for name in output_names]
leaked = [name for name in bare_names if name in EXCLUDED_COLUMNS]
if leaked:
    raise ValueError(f"Llegaron columnas excluidas a la salida: {leaked}")

pd.testing.assert_frame_equal(X_train, train_before_fit)
print("Sin NaN ni infinitos. Ninguna columna excluida está en la salida. X_train sigue intacto.")
print("Muestra sin escalar (tres filas de train):")
unscaled_sample


Filas de entrada: 283147
Filas de salida: 283147
Columnas crudas de entrada: 9
Features de salida: 13
Nombres: ['numeric__n_chords', 'numeric__log1p_n_chords', 'numeric__n_unique_chords', 'numeric__unique_ratio', 'numeric__share_major', 'numeric__share_minor', 'numeric__share_power', 'numeric__share_suspended', 'numeric__share_diminished', 'numeric__share_augmented', 'numeric__share_unknown', 'numeric__abab_share', 'binary__has_structure']


Sin NaN ni infinitos. Ninguna columna excluida está en la salida. X_train sigue intacto.
Muestra sin escalar (tres filas de train):


,n_chords,log1p_n_chords,n_unique_chords,unique_ratio,share_major,share_minor,share_power,share_suspended,share_diminished,share_augmented,share_unknown,abab_share,has_structure
0,67.0,4.219508,11.0,0.164179,0.910448,0.089552,0.0,0.0,0.0,0.0,0.0,0.062500,1.0
1,122.0,4.812184,6.0,0.049180,1.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.008403,1.0
2,56.0,4.043051,7.0,0.125000,0.464286,0.535714,0.0,0.0,0.0,0.0,0.0,0.301887,1.0


## 11. Lectura del resultado

La entrada de train tiene las 9 columnas que quedan tras separar el target. La salida tiene 13 números: 12 numéricos escalados y la bandera `has_structure` sin escalar.

Ese tamaño es el que el EDA sostiene. No aparece el vocabulario de acordes (miles de grafías) ni el de artistas. `log1p_n_chords` convive con `n_chords` por la razón ya dicha; no es una dimensión accidental.

Las proporciones de la muestra sin escalar se leen como fracción de los acordes de esa pista. `abab_share` es la fracción de ventanas de cuatro acordes que repiten el patrón. `has_structure` dice si el tab trae al menos un tag de parte, no si el estribillo suena distinto de la estrofa.

La imputación, ajustada con train, no tiene huecos que llenar en estas columnas. El escalado sí aprende media y desviación, y lo hace solo con train: un `transform` posterior del test usará esas estadísticas y no las recalculará.


## 12. Cierre para el notebook de modelado

**Decisiones cerradas.** Universo de 12 clases con `main_genre` presente. Split por `artist_id` (semilla 42, 20% de artistas). Features solo de `chords`: longitud y su `log1p`, acordes únicos y su razón, proporciones de calidad, `abab_share` y `has_structure`. Imputación como guarda y `StandardScaler` en las numéricas. `has_structure` se queda en 0/1.

**Excluidas.** `id`, `artist_id`, `spotify_song_id`, `spotify_artist_id`, `genres`, `rock_genre`, `release_date`, `decade` y el string `chords`.

**Limitaciones.** El EDA que motivó estas reglas vio también las filas que ahora son test. El test de este notebook no se usó para cambiarlas. Quedan 110 grupos (267 filas) con la misma progresión en artistas distintos; un split por artista no los separa todos. Pop, rock, pop rock, alternative y punk siguen compartiendo lenguaje armónico. Pop es cerca del 24% de las etiquetadas.

**Hipótesis que el modelado tiene que comparar, no esta entrega.** Bolsa de nombres frente a 4-gramas absolutos y frente a 4-gramas en grados de la escala, sin sustituir la grafía absoluta: el margen de tonalidad es menor que 0,05 en el 29,6% de las pistas. Macro-F1 junto a accuracy. Si un modelo lineal no gana nada con `n_chords` además de `log1p_n_chords`, quedarse con una de las dos.

**Objeto que sigue.** `pipeline` está ajustado solo con train y termina en `passthrough`. El siguiente notebook sustituye ese paso por un estimador y lo ajusta dentro de validación cruzada por artista. No se guardan matrices, modelos ni archivos binarios, y aquí no se busca hiperparámetros.
